In [229]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
import requests

from dotenv import load_dotenv

load_dotenv()

True

In [230]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [231]:
import os

EXCHANGE_RATE_API_KEY= os.getenv('EXCHANGE_RATE_API_KEY')

print(EXCHANGE_RATE_API_KEY)

566d2383234154630a5961a9


In [232]:
# tools creation

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """

    url = f'https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API_KEY}/pair/{base_currency}/{target_currency}'

    response = requests.get(url)
    return response.json()


In [233]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1787961601,
 'time_last_update_utc': 'Sat, 29 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1788048001,
 'time_next_update_utc': 'Sun, 30 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.5338}

In [234]:
@tool
def convert(base_currency_value: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value
    """

    return base_currency_value * conversion_rate

In [235]:
convert.invoke({'base_currency_value':10, 'conversion_rate':95})

950.0

### Binding the tool with LLM

In [236]:
llm = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-72B-Instruct',
    task='text-generation'
)
model = ChatHuggingFace(llm=llm)

llm_with_tools = model.bind_tools([get_conversion_factor, convert])

In [237]:
messages = [HumanMessage('What is the conversion factor between usd and inr. And what is the conversion factor between USD and GBP')]

In [238]:
messages

[HumanMessage(content='What is the conversion factor between usd and inr. And what is the conversion factor between USD and GBP', additional_kwargs={}, response_metadata={})]

In [239]:
ai_message = llm_with_tools.invoke(messages)
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_T11TFrPYNmareVt7w5wQoK3E', 'type': 'function'}, {'function': {'arguments': '{"base_currency": "USD", "target_currency": "GBP"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_YdOJk10MzxwOlm2N6TxEXn2d', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 442, 'total_tokens': 501}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04e50-a0b9-79a2-9e79-25725c684d73-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_T11TFrPYNmareVt7w5wQoK3E', 'type': 'tool_call'}, {'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'GBP'}, 'id': 'call

In [240]:
messages.append(ai_message)

In [241]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_T11TFrPYNmareVt7w5wQoK3E',
  'type': 'tool_call'},
 {'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'GBP'},
  'id': 'call_YdOJk10MzxwOlm2N6TxEXn2d',
  'type': 'tool_call'}]

In [242]:
import json

for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        #fetch this conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        # append this tool message to messages list
        messages.append(tool_message1)
    # execute the 2nd took using the conversion rate from tool 1
    if tool_call['name'] == 'convert':
        #fetch the current arg
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

   

In [243]:
messages

[HumanMessage(content='What is the conversion factor between usd and inr. And what is the conversion factor between USD and GBP', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_T11TFrPYNmareVt7w5wQoK3E', 'type': 'function'}, {'function': {'arguments': '{"base_currency": "USD", "target_currency": "GBP"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_YdOJk10MzxwOlm2N6TxEXn2d', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 442, 'total_tokens': 501}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04e50-a0b9-79a2-9e79-25725c684d73-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency'

In [244]:
llm_with_tools.invoke(messages)

AIMessage(content='The conversion factor between USD and INR is 95.5338, and the conversion factor between USD and GBP is 0.7379.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 935, 'total_tokens': 970}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04e50-ab79-7d10-88c3-8c74c1a8a931-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 935, 'output_tokens': 35, 'total_tokens': 970})